# Notebook 05: Collaborative Filtering & Popularity Baselines


In [1]:
import os
import sys
import time
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
import joblib

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

print("Collaborative Filtering libraries loaded successfully!")


Collaborative Filtering libraries loaded successfully!


## Section 1: Load Temporal Partitions & Catalog

### What we are doing:
We load the 14-week training partition and 1-week validation partition prepared in Notebook 03.


In [2]:
PROJECT_ROOT = Path(".").resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_MODELS_DIR = PROJECT_ROOT / "artifacts" / "models"
ARTIFACTS_META_DIR = PROJECT_ROOT / "artifacts" / "metadata"

# Load training and validation partitions
df_train = pd.read_parquet(DATA_PROCESSED_DIR / "transactions_train_14w.parquet")
df_val = pd.read_parquet(DATA_PROCESSED_DIR / "transactions_val_1w.parquet")
df_articles = pd.read_parquet(DATA_PROCESSED_DIR / "hm_articles_cleaned.parquet")

print("Loaded Temporal Partitions:")
print(f"  - Train Set (14 Weeks): {len(df_train):,} interactions ({df_train['t_dat'].min()} to {df_train['t_dat'].max()})")
print(f"  - Val Set (1 Week):     {len(df_val):,} interactions ({df_val['t_dat'].min()} to {df_val['t_dat'].max()})")
print(f"  - Catalog Articles:     {len(df_articles):,} items")

# Group validation interactions by customer for fast evaluation
val_ground_truth = df_val.groupby("customer_id")["article_id"].apply(set).to_dict()
print(f"  - Active Validation Customers: {len(val_ground_truth):,}")


Loaded Temporal Partitions:
  - Train Set (14 Weeks): 4,018,068 interactions (2020-06-02 to 2020-09-07)
  - Val Set (1 Week):     229,082 interactions (2020-09-08 to 2020-09-14)
  - Catalog Articles:     105,542 items


  - Active Validation Customers: 55,191


## Section 2: Model 1 — Recency-Weighted Popularity Baseline


In [3]:
# Extract top popular items in recent training weeks
max_train_date = pd.to_datetime(df_train["t_dat"].max())
two_weeks_prior = (max_train_date - pd.Timedelta(weeks=2)).strftime("%Y-%m-%d")

recent_train = df_train[df_train["t_dat"] >= two_weeks_prior]
popular_articles = recent_train["article_id"].value_counts().head(50).index.tolist()

print(f"Identified top {len(popular_articles)} popular articles in recent 2 weeks of training.")
print("Top 5 popular items:", popular_articles[:5])

class PopularityRecommender:
    def __init__(self, popular_items):
        self.popular_items = popular_items
        
    def recommend(self, customer_id, top_k=12):
        return self.popular_items[:top_k]

pop_model = PopularityRecommender(popular_articles)


Identified top 50 popular articles in recent 2 weeks of training.
Top 5 popular items: ['0751471001', '0915529003', '0751471043', '0706016001', '0915526001']


## Section 3: Model 2 — Item-Item Collaborative Filtering (Cosine Co-Occurrence)


In [4]:
# Build contiguous user and item mappings
train_users = sorted(df_train["customer_id"].unique())
train_items = sorted(df_train["article_id"].unique())

user2idx = {u: i for i, u in enumerate(train_users)}
item2idx = {it: i for i, it in enumerate(train_items)}
idx2item = {i: it for i, it in enumerate(train_items)}

# Map train transactions to indices
row_indices = df_train["customer_id"].map(user2idx).values
col_indices = df_train["article_id"].map(item2idx).values
values = np.ones(len(df_train), dtype=np.float32)

# Construct user-item sparse matrix (U x I)
n_users = len(train_users)
n_items = len(train_items)
R = sp.csr_matrix((values, (row_indices, col_indices)), shape=(n_users, n_items))
R.data = np.ones_like(R.data) # Binarize multiple purchases
print(f"User-Item Sparse Matrix R: {R.shape[0]:,} users x {R.shape[1]:,} items (Non-zeros: {R.nnz:,})")

# Compute Item-Item Co-occurrence: S = R^T * R (I x I)
t0 = time.time()
item_cooccur = R.T.dot(R)
# Zero out diagonal
item_cooccur.setdiag(0)
item_cooccur.eliminate_zeros()

# Normalize by item degrees: S_norm = D^-0.5 * S * D^-0.5
item_degrees = np.sqrt(np.asarray(R.sum(axis=0)).ravel())
item_degrees[item_degrees == 0] = 1.0
D_inv = sp.diags(1.0 / item_degrees)
S_norm = D_inv.dot(item_cooccur).dot(D_inv).tocsr()

print(f"Item-Item Cosine Similarity Matrix S computed in {time.time()-t0:.2f}s: shape {S_norm.shape}")

# Precompute customer purchase history dictionary
customer_history = df_train.groupby("customer_id")["article_id"].apply(set).to_dict()

class ItemItemCFRecommender:
    def __init__(self, S_matrix, item2idx, idx2item, customer_history, fallback_popular):
        self.S = S_matrix
        self.item2idx = item2idx
        self.idx2item = idx2item
        self.history = customer_history
        self.fallback = fallback_popular
        
    def recommend(self, customer_id, top_k=12):
        if customer_id not in self.history:
            return self.fallback[:top_k]
            
        purchased = [self.item2idx[it] for it in self.history[customer_id] if it in self.item2idx]
        if not purchased:
            return self.fallback[:top_k]
            
        # Aggregate similarity: sum of rows of S corresponding to purchased items
        user_scores = np.asarray(self.S[purchased].sum(axis=0)).ravel()
        
        # Zero out already purchased items
        user_scores[purchased] = -1.0
        
        # Top-K item indices
        top_indices = np.argsort(user_scores)[::-1][:top_k]
        recs = [self.idx2item[i] for i in top_indices if user_scores[i] > 0]
        
        # Fill remaining slots with popular items if needed
        if len(recs) < top_k:
            for it in self.fallback:
                if it not in recs and it not in self.history[customer_id]:
                    recs.append(it)
                if len(recs) == top_k:
                    break
        return recs

item_cf_model = ItemItemCFRecommender(S_norm, item2idx, idx2item, customer_history, popular_articles)
print("ItemItemCFRecommender initialized successfully!")


User-Item Sparse Matrix R: 322,623 users x 42,919 items (Non-zeros: 3,510,598)


Item-Item Cosine Similarity Matrix S computed in 11.63s: shape (42919, 42919)


ItemItemCFRecommender initialized successfully!


## Section 4: Model 3 — Implicit Matrix Factorization (Latent Factors)


In [5]:
# Fit 64-dimensional latent factor model
n_components = 64
print(f"Fitting TruncatedSVD with {n_components} latent factors...")
t0 = time.time()

svd = TruncatedSVD(n_components=n_components, random_state=42, algorithm="randomized")
user_factors = svd.fit_transform(R)
item_factors = svd.components_.T # Shape: (n_items, 64)

print(f"Matrix Factorization completed in {time.time()-t0:.2f}s.")
print(f"User Factors shape: {user_factors.shape}")
print(f"Item Factors shape: {item_factors.shape}")
print(f"Explained Variance Ratio sum: {svd.explained_variance_ratio_.sum():.4f}")

# Normalize factors for cosine dot-product
user_norms = np.linalg.norm(user_factors, axis=1, keepdims=True)
user_norms[user_norms == 0] = 1.0
user_factors_norm = user_factors / user_norms

item_norms = np.linalg.norm(item_factors, axis=1, keepdims=True)
item_norms[item_norms == 0] = 1.0
item_factors_norm = item_factors / item_norms

# Save trained factor matrices
np.save(ARTIFACTS_MODELS_DIR / "cf_user_factors.npy", user_factors_norm)
np.save(ARTIFACTS_MODELS_DIR / "cf_item_factors.npy", item_factors_norm)
joblib.dump({"user2idx": user2idx, "item2idx": item2idx, "idx2item": idx2item}, ARTIFACTS_MODELS_DIR / "cf_mappings.joblib")
print("Saved CF latent factors and mappings to artifacts/models/")

class MatrixFactorizationRecommender:
    def __init__(self, user_factors, item_factors, user2idx, idx2item, customer_history, fallback):
        self.user_factors = user_factors
        self.item_factors = item_factors
        self.user2idx = user2idx
        self.idx2item = idx2item
        self.history = customer_history
        self.fallback = fallback
        
    def recommend(self, customer_id, top_k=12):
        if customer_id not in self.user2idx:
            return self.fallback[:top_k]
            
        u_idx = self.user2idx[customer_id]
        u_vec = self.user_factors[u_idx] # (64,)
        
        # Dot product against all items: (64,) * (n_items, 64)^T -> (n_items,)
        scores = np.dot(self.item_factors, u_vec)
        
        # Mask out previously purchased items
        if customer_id in self.history:
            purchased_indices = [user2idx[customer_id]] # fast mask
            for it in self.history[customer_id]:
                if it in item2idx:
                    scores[item2idx[it]] = -1.0
                    
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.idx2item[i] for i in top_indices]

mf_model = MatrixFactorizationRecommender(user_factors_norm, item_factors_norm, user2idx, idx2item, customer_history, popular_articles)
print("MatrixFactorizationRecommender initialized successfully!")


Fitting TruncatedSVD with 64 latent factors...


Matrix Factorization completed in 10.74s.
User Factors shape: (322623, 64)
Item Factors shape: (42919, 64)
Explained Variance Ratio sum: 0.0518


Saved CF latent factors and mappings to artifacts/models/
MatrixFactorizationRecommender initialized successfully!


In [6]:
# Evaluation function
def evaluate_recommender(recommender_model, val_gt, sample_size=5000, top_k=12):
    np.random.seed(42)
    val_users = list(val_gt.keys())
    sample_users = np.random.choice(val_users, size=min(sample_size, len(val_users)), replace=False)
    
    hits = 0
    precisions = []
    recalls = []
    maps = []
    
    for uid in sample_users:
        actual = val_gt[uid]
        preds = recommender_model.recommend(uid, top_k=top_k)
        
        # Hit Rate
        hit = len(set(preds) & actual) > 0
        hits += int(hit)
        
        # Precision & Recall
        intersection = len(set(preds) & actual)
        precisions.append(intersection / top_k)
        recalls.append(intersection / len(actual))
        
        # Average Precision (MAP)
        score = 0.0
        num_hits = 0.0
        for i, p in enumerate(preds):
            if p in actual and p not in preds[:i]:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
        maps.append(score / min(len(actual), top_k))
        
    return {
        "HitRate@12": round(hits / len(sample_users) * 100, 2),
        "Precision@12": round(np.mean(precisions) * 100, 3),
        "Recall@12": round(np.mean(recalls) * 100, 3),
        "MAP@12": round(np.mean(maps) * 100, 3)
    }

print("Evaluating Models on Held-Out Week 15 Validation Set (5,000 Sampled Customers)...")
t0 = time.time()
pop_metrics = evaluate_recommender(pop_model, val_ground_truth)
print(f"Popularity Baseline:  {pop_metrics} ({time.time()-t0:.1f}s)")

t0 = time.time()
cf_metrics = evaluate_recommender(item_cf_model, val_ground_truth)
print(f"Item-Item CF Model:   {cf_metrics} ({time.time()-t0:.1f}s)")

t0 = time.time()
mf_metrics = evaluate_recommender(mf_model, val_ground_truth)
print(f"Matrix Factorization: {mf_metrics} ({time.time()-t0:.1f}s)")

# Display comparison table
df_comparison = pd.DataFrame([
    {"Model": "Popularity Baseline", **pop_metrics},
    {"Model": "Item-Item Collaborative Filtering", **cf_metrics},
    {"Model": "Matrix Factorization (64 Factors)", **mf_metrics}
])
display(df_comparison)

# Save comparison results
df_comparison.to_json(ARTIFACTS_META_DIR / "cf_validation_comparison.json", orient="records", indent=2)
print("Saved CF validation metrics to artifacts/metadata/cf_validation_comparison.json")


Evaluating Models on Held-Out Week 15 Validation Set (5,000 Sampled Customers)...
Popularity Baseline:  {'HitRate@12': 6.24, 'Precision@12': np.float64(0.552), 'Recall@12': np.float64(2.109), 'MAP@12': np.float64(0.575)} (0.0s)


Item-Item CF Model:   {'HitRate@12': 4.92, 'Precision@12': np.float64(0.472), 'Recall@12': np.float64(1.949), 'MAP@12': np.float64(0.747)} (12.8s)


Matrix Factorization: {'HitRate@12': 2.22, 'Precision@12': np.float64(0.205), 'Recall@12': np.float64(0.726), 'MAP@12': np.float64(0.25)} (10.6s)


,Model,HitRate@12,Precision@12,Recall@12,MAP@12
0,Popularity Baseline,6.2400,0.5520,2.1090,0.5750
1,Item-Item Collaborative Filtering,4.9200,0.4720,1.9490,0.7470
2,Matrix Factorization (64 Factors),2.2200,0.2050,0.7260,0.2500


Saved CF validation metrics to artifacts/metadata/cf_validation_comparison.json


## Section 6: Validation Assertions & Stage Sign-Off

### What we are doing:
We run formal assertions verifying that:
1. All models produce strictly top-12 recommendations.
2. Latent factor matrices and lookup mappings exist on disk.
3. Collaborative filtering outperforms random chance.


In [7]:
# Formal Validation Checks
print("Executing Collaborative Filtering Validation Assertions...")

# 1. Output shape check
test_user = list(val_ground_truth.keys())[0]
assert len(pop_model.recommend(test_user, top_k=12)) == 12, "Popularity returned wrong count"
assert len(item_cf_model.recommend(test_user, top_k=12)) == 12, "Item-CF returned wrong count"
assert len(mf_model.recommend(test_user, top_k=12)) == 12, "MF returned wrong count"

# 2. Metric validity
assert pop_metrics["HitRate@12"] > 0, "Popularity HitRate must be positive"
assert cf_metrics["HitRate@12"] > 0, "CF HitRate must be positive"

# 3. Artifact presence
assert (ARTIFACTS_MODELS_DIR / "cf_user_factors.npy").exists(), "User factors file missing"
assert (ARTIFACTS_MODELS_DIR / "cf_item_factors.npy").exists(), "Item factors file missing"

print("=" * 60)
print("ALL COLLABORATIVE FILTERING ASSERTIONS PASSED (100% OK)")
print("Notebook 05 execution complete. Ready for Notebook 06: Image-Based Recommendation.")
print("=" * 60)


Executing Collaborative Filtering Validation Assertions...
ALL COLLABORATIVE FILTERING ASSERTIONS PASSED (100% OK)
Notebook 05 execution complete. Ready for Notebook 06: Image-Based Recommendation.
